In [ ]:
"""
  Author : Margaret Woon
  Date : 2023/05/16
  version : 00001 - RUN 1 (Total 2 run)
  Working copy to extract confluence spec doc columns table
  Pre-cursor : "confluenceSpecDocConfig.txt" must present where this program is run
  Pre-cursor : "bancs_bronze_inct_url" is generated by this run
  Step 1 : Job proceed
  Step 2 : Output BaNCS_spec_doc_YYYYMMDD.xlsx is created
  
"""

import requests 
import pandas as pd
from bs4 import BeautifulSoup
from requests.auth import HTTPBasicAuth
from datetime import datetime
import json
import os


last_cnt = 0

domain_name = 'https://cua.atlassian.net'

target_url = "https://cua.atlassian.net/wiki/spaces/DP/pages/1888124933/INCT+var+area"

#####


####
class HTMLHrefParser:
        
    
    def __init__(self, url, my_config_js, my_href_cnt) :
        self.url = url
        self.config_js = my_config_js
        self.extract_cnt = my_href_cnt
        
         
        ## retrieve the proxy from the configuration file
        http_proxy = self.config_js['proxy']
        
        https_proxy = self.config_js['proxy']
        
        

    
        proxyDict ={"http": http_proxy,"https": https_proxy, "authenticate":True,"href": self.config_js['href']}

        # retrieve the email from the configuration file
        EMAIL = self.config_js['email']
              

        my_auth = HTTPBasicAuth(EMAIL, self.config_js['api_token'])
        
                
        try :
            response = requests.get(self.url, auth=my_auth, proxies=proxyDict)
        except Exception as e :
            print(e)
        
            print("\nFail to request data for url: {}".format(self.url))
        else :
            print(f"\nRequest to url {self.url} succeeded.")
        
        ### replace <br> tag with carriage return "\n"
        
        
        txt = ''
        
        txt = response.text.replace("<br/>","\n")
        
        ### Calling soup
        soup = BeautifulSoup(txt, 'lxml')
        
             
        
        page_title = soup.title.get_text()
        title_name_array = page_title.split(" ")
        #dataset_name = title_name_array[1]
                
        cnt = 0
        extract_table = self.config_js['confluence_table']

        
        href_cnt = 0
        df_url_dict = {}
        df_url = pd.DataFrame()
        for a_href in soup.find_all("a", href=True):
            
            print("url : {}".format(domain_name+a_href["href"]))
            
            ## find the last occurrence of "+" within a string
            table_end_pos = a_href["href"].find("/")
            
#             if "INCT+Transaction+Type" in a_href["href"]:
#                 href_cnt += 1
                
#                 target = a_href["href"]
#                 start_pos = a_href["href"].rfind("+")
#                 end_pos = len(a_href["href"])
                
#                 ### find the last occurrence of "/" within a string                  
#                 table_start_pos = a_href["href"].rfind("/")
                
                
#                 ### find the last occurrence of "+" within a string
#                 table_end_pos = a_href["href"].find("+")
                
#                 print("Dataset name : {}\n".format(target[table_start_pos+1:table_end_pos]+"_"+target[start_pos+1:]))
                
                
#                 ### construct the full url
#                 ds_url = domain_name+a_href["href"]
                
                
#                 ### identify the dataset name
#                 ds_name = target[table_start_pos+1:table_end_pos]+"_"+target[start_pos+1:]
#                 df_url_dict[ds_url]= ds_name
                
                                       

#         df_url = pd.DataFrame.from_dict(df_url_dict, orient = 'index')
        
#         ## add a new "index" column
#         df_url["index"] = df_url.index
        
#         ### rename columns 0 as "Dataset_name" and index as "Bancs_url"
#         df_url.rename(columns = {0:"Dataset_name","index":"Bancs_url"}, inplace=True)
        
#          ### reorder the columns
#         df_url = df_url.reindex(columns =["Bancs_url", "Dataset_name"])
        
#         ### output the dataframe into a csv file
#         df_url.to_csv(os.getcwd()+"\\bancs_bronze_inct_url.csv", index = None)
        return 
                        
                        
                        
                        
                        
##############################
###### Main Program
##############################

now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")

now_yyyymmdd = now.strftime("%Y%m%d")

print("Job proceed at {}".format(now_yyyymmdd_hhmmss))



## Open the configuration file in the current directory as read mode
with open(os.getcwd()+"\\confluenceSpecDocConfig.txt", "r") as f:
    confg_data = f.read()
    
f.close()

#### Load the json file into a dictionary
my_config_js = json.loads(confg_data)

print("Configuration file loaded successfully!\n\n")    

#df_url = pd.read_csv(os.getcwd()+"\\bancs_bronze_boct_url.csv")



print("\nProcessing url {}".format(target_url))
cnt = 0
tp = HTMLHrefParser(target_url, my_config_js, cnt)


now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")
print("\n\nJob completed at {}".format(now_yyyymmdd_hhmmss))


In [ ]:
"""
  Author : Margaret Woon
  Date : 2023/05/16
  version : 00001 - RUN 2
  Working copy to extract confluence spec doc columns table
  Pre-cursor : "confluenceSpecDocConfig.txt" must present where this program is run
  Pre-cursor : "bancs_bronze_url.csv" must present where this program is run
  Step 1 : Job proceed
  Step 2 : Output BaNCS_spec_doc_YYYYMMDD.xlsx is created
  
"""

import requests 
import pandas as pd
from bs4 import BeautifulSoup
from requests.auth import HTTPBasicAuth
from datetime import datetime
import json
import os

filename = 'BaNCS_Spec_Doc_INCT'
url_list = 'bancs_bronze_inct_url'

last_cnt = 0

#####


####
class HTMLTableParser:
        
    
    def __init__(self, url, my_config_js, my_table_name, my_last_cnt) :
        self.url = url
        self.config_js = my_config_js
        self.dataset_name = my_table_name
        self.extract_cnt = my_last_cnt
        
        
    ### this function is being called within the class    
    

        
    def parse_url(self):
        
        
        ### this function is being called within the class    
        def create_df(table_in):

            n_columns = 0
            n_rows=0
            column_names = []

           
            # Find number of rows and columns
            # we also find the column titles if we can
            for row in table_in.find_all('tr'):
                
                # Determine the number of rows in the table
                td_tags = row.find_all('td')
                if len(td_tags) > 0:
                    n_rows+=1
                    if n_columns == 0:
                        # Set the number of columns for our table
                        n_columns = len(td_tags)

                # Handle column names if we find them
                th_tags = row.find_all('th') 
                if len(th_tags) > 0 and len(column_names) == 0:
                    for th in th_tags:
                        column_names.append(th.get_text())

            # Safeguard on Column Titles
            if len(column_names) > 0 and len(column_names) != n_columns:
                raise Exception("Column titles do not match the number of columns")

            columns = column_names if len(column_names) > 0 else range(0,n_columns)
            df = pd.DataFrame(columns = columns,
                              index= range(0,n_rows))
            row_marker = 0
            for row in table_in.find_all('tr'):
                column_marker = 0
                columns = row.find_all('td')
                for column in columns:
                    df.iat[row_marker,column_marker] = column.get_text()
                    #print("columns text nnnnnnnnnnnnnnnnn ",column.get_text() )
                    column_marker += 1
                if len(columns) > 0:
                    row_marker += 1

            # Convert to float if possible
            for col in df:
                try:
                    df[col] = df[col].astype(float)
                except ValueError:
                    pass

            return df
    
        #### line marker
        
        ## retrieve the proxy from the configuration file
        http_proxy = self.config_js['proxy']
        
        https_proxy = self.config_js['proxy']
        
        

    
        proxyDict ={"http": http_proxy,"https": https_proxy, "authenticate":True,"href": self.config_js['href']}

        # retrieve the email from the configuration file
        EMAIL = self.config_js['email']
              

        my_auth = HTTPBasicAuth(EMAIL, self.config_js['api_token'])
        
        try :
            response = requests.get(self.url, auth=my_auth, proxies=proxyDict)
        except Exception as e :
            print(e)
        
            print("\nFail to request data for url: {}".format(self.url))
        else :
            print(f"\nRequest to url {self.url} succeeded.")
        
        ### replace <br> tag with carriage return "\n"
        
        
        txt = ''
        txt = response.text.replace("<br/>","\n")
        
        ### replace </p> tag with carriage return "\n\n"
        ###
        ###
        ###
        ###txt = txt.replace("</p>","</p> \n\n")       
        
        ### Calling soup
        soup = BeautifulSoup(txt, 'lxml')
       
        
        page_title = soup.title.get_text()
        title_name_array = page_title.split(" ")
        #dataset_name = title_name_array[1]
                
        cnt = 0
        extract_table = self.config_js['confluence_table']
        ### exception handing for "DEPP" page
        if self.dataset_name == 'DEPP':
             extract_table = 6
            
        print("\nStart processing table {}.".format(self.dataset_name))
        
        for table in soup.find_all('table'):
            cnt += 1
            #print(f"table {cnt} &&&&&&&&&&&&&       ", table, "\n\n\n\n")
            
            ### confluence table number 5 is the data dictionary data due to the of the template; this is configurable in config file                          
            if cnt == extract_table: 
                
                #print("table  data @@@@@@@@@  ", table )
                df = create_df(table)
                print(f"\nSpec Doc Dataframe creation for {self.dataset_name} succeeded.\n")
                
                if os.path.exists(excel_filename):
                    try :
                        with pd.ExcelWriter(os.getcwd()+"\\"+excel_filename, engine='openpyxl', mode='a') as writer:  
                            df.to_excel(writer, sheet_name=self.dataset_name, index = None)
                    except :
                         print(f"Fail to append tab {self.dataset_name} to excel.")
                    else :
                        print(f"{self.dataset_name} tab append to excel succeeded.")
                        self.extract_cnt += 1
                        
                else:
                    try :
                        with pd.ExcelWriter(os.getcwd()+"\\"+excel_filename) as writer:  
                            df.to_excel(writer, sheet_name=self.dataset_name, index = None)
                    except :
                        print(f"Fail to append tab {dataset_name} to excel.")
                    else :
                        print(f"{self.dataset_name} tab append to excel succeeded.")
                        self.extract_cnt += 1
        
        return self.extract_cnt
                        
                        
                        
                        
                        
##############################
###### Main Program
##############################

now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")

now_yyyymmdd = now.strftime("%Y%m%d")

print("Job proceed at {}".format(now_yyyymmdd_hhmmss))

excel_filename = filename+'_'+now_yyyymmdd+'.xlsx'

## Open the configuration file in the current directory as read mode
with open(os.getcwd()+"\\confluenceSpecDocConfig.txt", "r") as f:
    confg_data = f.read()
    
f.close()

#### Load the json file into a dictionary
my_config_js = json.loads(confg_data)

print("\nConfiguration file loaded successfully!\n")    

df_url = pd.read_csv(os.getcwd()+"\\"+url_list+".csv")

for i in range(len(df_url)):
    print("\nProcessing url {}".format(df_url.loc[i,"Bancs_url"]))
    target_url = df_url.loc[i,"Bancs_url"]
    table_name = df_url.loc[i,"Dataset_name"]
    tp = HTMLTableParser(target_url, my_config_js, table_name, last_cnt)
    last_cnt = tp.parse_url()
    print("*"*120)



print("\nConfiguration url count: {}".format(len(df_url)))
print("\nExtract count: {}".format(last_cnt))


now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")
print("\nJob completed at {}".format(now_yyyymmdd_hhmmss))